# 01 · What a tensor is

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/01-what-a-tensor-is.ipynb)

*Part I · demo · 20 min*

> 🇪🇸 **Qué es un tensor** — El vocabulario, la forma en NumPy y las tres operaciones que importan.

The vocabulary, shape in NumPy, and the three operations that matter.

## What you will be able to do

- Use the vocabulary: order, axis, mode, shape, slice, fiber, unfolding, contraction, decomposition.
- Read `.shape`, `.ndim` and `.size` off any array and say what each axis means.
- Take slices and fibers, and unfold a tensor into a matrix without losing anything.
- Write a dot product and a matrix product as `einsum` contractions.
- Place LU, QR, eigendecomposition, SVD, the pseudoinverse and Tucker in one map.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from skimage import data
from scipy.linalg import lu

rng = np.random.default_rng(0)

## 1.1 Vocabulary

> 🇪🇸 El vocabulario. Casi todos los términos son casi idénticos en español.

Keep this table open for the whole workshop.

| Term | Plain meaning | Spanish | Example |
|---|---|---|---|
| **Tensor** | An array of numbers with any number of axes | *tensor* | A colour image |
| **Axis** (pl. axes) | One direction along which data is arranged | *eje* | Height; width; colour |
| **Mode** | Another word for axis, used in tensor theory | *modo* | "mode-0 unfolding" |
| **Order** | How many axes a tensor has | *orden* | A matrix has order 2 |
| **Shape** | The size along each axis, as a tuple | *forma* | `(512, 512, 3)` |
| **Slice** | Fix one index, keep the rest | *corte* | One colour channel |
| **Fiber** | Fix every index except one | *fibra* | The 3 colour values of one pixel |
| **Unfolding** | Rearranging a tensor into a matrix | *desplegado* | Needed for decompositions |
| **Contraction** | Multiply and sum over a shared axis | *contracción* | The dot product |
| **Decomposition** | Writing one tensor as a product of simpler ones | *descomposición* | SVD, Tucker |

⚠️ **Warning about the word "rank".** In Chapter 2, *rank* means the number of
independent columns of a matrix. In tensor theory, *rank* often means the number
of axes. To avoid confusion, this workshop says **order** for the number of
axes, and **rank** only in Chapter 2's sense.

## 1.2 Shape in NumPy

> 🇪🇸 La forma en NumPy: `.shape`, `.ndim` y `.size`.

Every NumPy array has `.shape`, a tuple giving the size along each axis. The
length of that tuple is `.ndim`, the number of axes.

In [ ]:
scalar = np.array(3.0)                     # book: a           — order 0
vector = np.array([1., 2., 3.])            # book: x, x_i      — order 1
matrix = np.array([[1., 2.], [3., 4.]])    # book: A, A_{i,j}  — order 2
tensor = rng.standard_normal((2, 3, 4))    # book: A_{i,j,k}   — order 3

for name, arr in [("scalar", scalar), ("vector", vector),
                  ("matrix", matrix), ("tensor", tensor)]:
    print(f"{name:8s} shape={str(arr.shape):12s} ndim={arr.ndim}  size={arr.size}")

A scalar has `shape=()`, an empty tuple — there are no axes to measure. And
`size` is always the product of the numbers in `shape`: 2 × 3 × 4 = 24.

Now with real data.

In [ ]:
digits = load_digits()
print(digits.images.shape)      # (1797, 8, 8)  — 1797 handwritten digits, 8x8 pixels

photo = data.immunohistochemistry()
print(photo.shape)              # (512, 512, 3) — height, width, colour

Both are order 3, but their axes mean completely different things.
`digits.images` counts *images* along axis 0; `photo` counts *colours* along
axis 2. **The shape alone never tells you what the axes mean.** You must know,
and you must keep track.

## Exercise 1 — read the shapes

> 🇪🇸 Lee las formas y di qué significa cada eje.

In [ ]:
# TODO 1: Build a scalar, a vector, a matrix and an order-3 tensor, and print
#         .shape, .ndim and .size for each. Which one has shape ()?

# TODO 2: Take load_digits().images and data.astronaut(). Both are order 3.
#         For each, write down in a comment what axis 0, 1 and 2 count.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
for arr in [np.array(3.0), np.zeros(3), np.zeros((2, 2)), np.zeros((2, 3, 4))]:
    print(arr.shape, arr.ndim, arr.size)
# ()        0 1
# (3,)      1 3
# (2, 2)    2 4
# (2, 3, 4) 3 24

print(load_digits().images.shape)   # (1797, 8, 8)   axis 0 = which image
                                    #                axis 1 = row of pixels
                                    #                axis 2 = column of pixels
print(data.astronaut().shape)       # (512, 512, 3)  axis 0 = height
                                    #                axis 1 = width
                                    #                axis 2 = colour channel

## 1.3 The three operations that matter

> 🇪🇸 Cortes y fibras, desplegado y contracción — las tres operaciones clave.

### Slices and fibers — fixing indices takes a tensor apart

In [ ]:
print(photo[:, :, 0].shape)       # (512, 512) — a slice: one colour channel, still an image
print(photo[100, 200, :].shape)   # (3,)       — a fiber: the 3 colour values of one pixel

### Unfolding — every decomposition begins here

Every tensor decomposition begins by turning the tensor into a matrix, one axis
at a time. Move axis *k* to the front, then flatten everything else into one
long axis.

In [ ]:
def unfold(T, axis):
    """Move `axis` to the front, flatten everything else into one long axis."""
    return np.moveaxis(T, axis, 0).reshape(T.shape[axis], -1)

print(unfold(photo, 0).shape)   # (512, 1536) — rows are the height axis
print(unfold(photo, 2).shape)   # (3, 262144) — rows are the 3 colour channels

Unfolding **loses nothing**. It only rearranges. The mode-2 unfolding says
"each colour channel is one row of 262,144 numbers" — and now every matrix tool
you know, including SVD, can be applied to it.

You will use this exact function again in sections 07 and 10.

### Contraction — multiply along a shared axis and sum over it

The dot product (eq. 2.8) and the matrix product (eq. 2.5) are both
contractions. `np.einsum` writes them directly.

In [ ]:
a = np.array([1., 2., 3.]); b = np.array([4., 5., 6.])
print(np.einsum('i,i->', a, b))          # dot product, sum over i          (eq 2.8)

A = np.array([[1., 2.], [3., 4.]]); B = np.array([[5., 6.], [7., 8.]])
print(np.einsum('ik,kj->ij', A, B))      # matrix product, sum over k       (eq 2.5)

**The rule, in one sentence:** an index that appears in the inputs but **not**
after the arrow is summed over; an index that appears after the arrow is kept.

That one sentence is the whole of section 06.

## Exercise 2 — take a tensor apart and put it back

> 🇪🇸 Desmonta un tensor y vuelve a montarlo.

In [ ]:
# TODO 3: From `photo`, extract (a) the green channel as a (512, 512) slice and
#         (b) the colour fiber at pixel (10, 20). Which is a slice, which a fiber?

# TODO 4: Unfold `photo` along all three axes and print the three shapes.
#         Confirm that each unfolding has exactly photo.size entries —
#         unfolding rearranges, it never loses anything.

# TODO 5: Write the dot product of `a` and `b` as einsum, and check it against
#         np.dot. Then write the matrix product of A and B, and check against @.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
green = photo[:, :, 1]        # slice — one index fixed, the rest kept
fiber = photo[10, 20, :]      # fiber — every index fixed except one
print(green.shape, fiber.shape)                # (512, 512) (3,)

for ax in range(3):
    M = unfold(photo, ax)
    print(ax, M.shape, M.size == photo.size)   # True every time

print(np.einsum('i,i->', a, b), np.dot(a, b))              # 32.0 32.0
print(np.allclose(np.einsum('ik,kj->ij', A, B), A @ B))    # True

## 1.4 The map of factorizations

> 🇪🇸 El mapa de las factorizaciones: qué método sirve para qué.

A **factorization** writes one object as a product of simpler objects. You met
two in Chapter 2. Here is the whole family we will use today.

| Method | Works on | What it gives you | Where today |
|---|---|---|---|
| **LU** | Square matrix | Gaussian elimination, saved for reuse | Below |
| **QR / Gram-Schmidt** | Any matrix | Perpendicular, unit-length directions | Below |
| **Eigendecomposition** | Square matrix | Directions that only get scaled (§2.7) | Section 08 |
| **SVD** | Any matrix | The most general matrix factorization (§2.8) | Sections 07 and 10 |
| **Pseudoinverse** | Any matrix | "Inverse" when no true inverse exists (§2.9) | Section 07 |
| **Tucker / CP** | **Tensor, any order** | PCA generalized to every axis | Section 10 |

In [ ]:
A3 = np.array([[4., 3., 2.], [2., 1., 1.], [6., 3., 5.]])

P, L, U = lu(A3)                            # LU: A = P L U
print(np.allclose(P @ L @ U, A3))           # True

Q, R = np.linalg.qr(A3)                     # QR: orthonormal directions
print(np.allclose(Q.T @ Q, np.eye(3)))      # True — book eq 2.37

**LU** is Gaussian elimination stored as two triangular matrices, so `Ax = b`
can be solved cheaply many times for different `b`. **QR** (computed by
Gram-Schmidt, or more stably by other methods) produces *orthonormal* directions
— mutually perpendicular, each of length 1. It is used for orthogonal weight
initialization in neural networks and for stable least squares.

Everything in that table except the last row works on **matrices** — two axes.
Real data often has more. That is what section 10 addresses.

## Exercise 3 — which factorization?

> 🇪🇸 ¿Qué factorización usarías en cada caso?

In [ ]:
# TODO 6: For each situation, name the method from the table above.
#         Write your answer as a comment — no code needed.
#           (a) You must solve Ax = b for 500 different b, with the same square A.
#           (b) You need mutually perpendicular, unit-length directions.
#           (c) A has more rows than columns and there is no exact solution.
#           (d) Your data has three axes and you want to compress all three.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
# (a) LU  — factor once, then each new b is two cheap triangular solves.
# (b) QR  — Q's columns are orthonormal (Q.T @ Q == I).
# (c) Pseudoinverse — section 07. It is built from the SVD.
# (d) Tucker — section 10. PCA can only ever see two axes.

---

## Done with this section

Next up: **02 · Thinking in N dimensions** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/02-thinking-in-n-dimensions.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)